<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F06_spark_multi.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. The Terraform-deployed templates already carry the `SF_*` run identity in their env,
> so there's no environment cell to fill in. Runs on the **`sf-main`** runtime template (Python 3.11). See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.


# 06 · Spark **multi** — one run, one child batch per family

`spark_method='multi'` fans a run out into **one child `explode` batch per model family** (statistical / ml / deep_learning / native) — separate autoscaling and failure domains — all under **one** shared `run_id` and **one** `run_registry` header (C3). A slow deep-learning family can't starve the statistical family's executors, yet the leaderboard still shows the whole run as a single `run_id` with every family under it.

Multi is orchestrated from the client by `submit.submit_multi` (not `main.run`, which runs one batch): `google-cloud-dataproc` — the `[spark]` extra — isn't in the runtime container, so the fan-out into child batches lives on the submitter, not on-cluster.

## Get the code (cloud runtimes only)

On a cloud notebook (Colab Enterprise, Vertex Workbench) this clones or updates the repo so you're on the latest `src/`. **Skip it in a local clone** — it's a no-op guarded on the package already being importable. `submit_multi` needs the `[spark]` extra in the kernel (`pip install -e 'scale-forecasting[spark]'`) for `google-cloud-dataproc`.

In [ ]:
# Cloud bootstrap: clone + editable-install so `import scale_forecasting` resolves.
# Harmless locally — if the package already imports, we do nothing.
import importlib.util
import os
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[spark]"], check=True)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))

## Resolve the deployment (live GCP)

`Settings.resolve()` reads the `SF_*` environment — the *same* identity every writer uses (G1), so a notebook run and a Composer run land in the same registry. Required: `SF_PROJECT_ID`, `SF_CONNECTION`, `SF_WAREHOUSE_URI`; `SF_DATASET_ID`/`SF_REGION` default. `submit_multi` also resolves `BatchInfra` from the same `SF_*`/`terraform output` env (compute SA, code bucket, container image, subnet).

In [ ]:
from google.cloud import bigquery

from scale_forecasting.settings import Settings

settings = Settings.resolve()
client = bigquery.Client(project=settings.project_id)
DATASET = settings.dataset_ref
print("deployment:", DATASET, "region:", settings.region)

## Review helpers

Registry rows written through the Storage Write API are *async-visible*, so we poll the leaderboard briefly until a run's models show up. `leaderboard(run_id)` returns one row per model — every family's models share the **one** `run_id` — and `run_summary(run_id)` is the header roll-up.

In [ ]:
import time

import pandas as pd


def _query(sql, run_id):
    job = client.query(
        sql,
        job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("run_id", "STRING", run_id)]
        ),
    )
    return job.result().to_dataframe()


def leaderboard(run_id, expect_models=None, tries=15, pause=4.0):
    """Poll v_model_leaderboard until `expect_models` all appear (or give up), newest metrics."""
    sql = (
        f"SELECT model_type, compute_engine, n_cells, mean_wape, mean_mae, median_fit_seconds "
        f"FROM `{DATASET}.v_model_leaderboard` WHERE run_id=@run_id "
        f"ORDER BY mean_wape"
    )
    df = pd.DataFrame()
    for _ in range(tries):
        df = _query(sql, run_id)
        if expect_models is None or set(df["model_type"]) >= set(expect_models):
            break
        time.sleep(pause)
    return df


def run_summary(run_id):
    sql = f"SELECT * FROM `{DATASET}.v_run_summary` WHERE run_id=@run_id"
    return _query(sql, run_id)

## Parameters — edit me

Everything that shapes the run is set here as plain Python, then assembled into a **`RunConfig`** — the one frozen object that drives the run and is logged verbatim to `run_registry.raw_config`, so *this cell is the experiment record* (G2/G3).

- **`MODELS`** deliberately spans two families — `theta`/`holtwinters` (**statistical**) and `xgboost` (**ml**) — so `submit_multi` fans out into **two** child `explode` batches, one per family. Each child autoscales and fails independently.
- **`spark_method='multi'`** is set on the config; `submit_multi` reads it and orchestrates the per-family children (there is no on-cluster `multi` engine — each child runs the ordinary `explode` engine on its family's model subset).
- **`SERIES_LIMIT=100`** — the C6 demo scale (the *same* first 100 series every approach uses, DESIGN §13.1).

In [ ]:
from scale_forecasting.config import RunConfig
from scale_forecasting.registry.ids import make_run_id

# === Parameters — edit me ===============================================
RUN_NAME = f"nb06 spark multi {int(time.time())}"  # timestamped → a fresh run each execution
SOURCE_TABLE = "source_series_iceberg"  # shipped seed (100k); _iceberg↔_native to compare storage
MODELS = ["theta", "holtwinters", "xgboost"]  # statistical (theta, holtwinters) + ml (xgboost)
SPARK_METHOD = "multi"  # one child explode batch per family, under one run_id
HORIZON = 28
SERIES_LIMIT = 100  # the C6 demo scale
HOLIDAYS = ["US"]
BACKTEST = True  # OOF metric panel so the leaderboard is scored (mean_wape / mean_mae)
N_FOLDS = 2
# ========================================================================

cfg = RunConfig(
    run_name=RUN_NAME,
    python_runtime="spark",
    spark_method=SPARK_METHOD,
    data={"source_table": SOURCE_TABLE, "horizon": HORIZON, "series_limit": SERIES_LIMIT},
    models=MODELS,
    features={"holidays": HOLIDAYS},
    backtest={"enabled": BACKTEST, "n_folds": N_FOLDS, "horizon": HORIZON, "step": HORIZON},
)
run_id = make_run_id(cfg)
print("run_id:", run_id, "| models:", cfg.models, "| method:", cfg.spark_method)

## Run — one child batch per family, one `run_id`

`submit_multi(cfg)` writes the shared header (RUNNING) up front, submits one `explode` child batch per family (each in contributor mode — `manage_header=False`, so no child touches the header), and finalizes the header COMPLETED iff every child succeeds. It returns the **child batch ids** — one per family — all under the single `run_id` above.

In [ ]:
from scale_forecasting.submit import submit_multi

child_batch_ids = submit_multi(cfg)
print("child batches (one per family):")
for bid in child_batch_ids:
    print("  ", bid)
print("all under one run_id:", run_id)

## Review — every family on one leaderboard, one `run_id`

All three models ran as Spark cells across two child batches, but the leaderboard shows them under the **one** shared `run_id` (post-C3: multi is one run, not one-per-family). `run_summary` confirms a single header covering every cell.

In [ ]:
board = leaderboard(run_id, expect_models=cfg.models)
board

In [ ]:
run_summary(run_id)